# Chapter 2: The Simple Regression Model

Covers simple OLS estimation, fitted values and residuals, goodness of fit (R²), nonlinearities, and Monte Carlo verification of OLS properties.

## Step 1 – Setup: Working Directory & Output Folder

Run this cell first to fix the working directory and create the `output/` folder.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import wooldridge as woo
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## 2.1 Simple OLS Regression

## Step 2 – The Simple Regression Model

The **population model** is:
$$y_i = \beta_0 + \beta_1 x_i + u_i, \quad i=1,\ldots,n$$

where $u_i$ is the error term capturing all factors other than $x$ that affect $y$.

**Gauss-Markov assumptions (SLR.1–SLR.5):**
1. Linear in parameters
2. Random sampling
3. Sample variation in $x$
4. Zero conditional mean: $E(u|x)=0$
5. Homoscedasticity: $\text{Var}(u|x)=\sigma^2$

**OLS estimators** minimise $\sum_i (y_i - \hat{\beta}_0 - \hat{\beta}_1 x_i)^2$:

$$\hat{\beta}_1 = \frac{\sum_{i=1}^n (x_i-\bar{x})(y_i-\bar{y})}{\sum_{i=1}^n (x_i-\bar{x})^2}= \frac{\widehat{\text{Cov}}(x,y)}{\widehat{\text{Var}}(x)}$$

$$\hat{\beta}_0 = \bar{y} - \hat{\beta}_1\bar{x}$$

In [4]:
df = woo.dataWoo('wage1')

# OLS via statsmodels
model = smf.ols('wage ~ educ', data=df).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                   wage   R-squared:                       0.165
Model:                            OLS   Adj. R-squared:                  0.163
Method:                 Least Squares   F-statistic:                     103.4
Date:                Fri, 13 Mar 2026   Prob (F-statistic):           2.78e-22
Time:                        10:11:51   Log-Likelihood:                -1385.7
No. Observations:                 526   AIC:                             2775.
Df Residuals:                     524   BIC:                             2784.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.9049      0.685     -1.321      0.1

In [5]:
df.head()

,wage,educ,exper,tenure,nonwhite,female,married,numdep,smsa,northcen,...,trcommpu,trade,services,profserv,profocc,clerocc,servocc,lwage,expersq,tenursq
0,3.10,11,2,0,0,1,0,2,1,0,...,0,0,0,0,0,0,0,1.131402,4,0
1,3.24,12,22,2,0,1,1,3,1,0,...,0,0,1,0,0,0,1,1.175573,484,4
2,3.00,11,2,0,0,0,0,2,0,0,...,0,1,0,0,0,0,0,1.098612,4,0
3,6.00,8,44,28,0,0,1,0,1,0,...,0,0,0,0,0,1,0,1.791759,1936,784
4,5.30,12,7,2,0,0,1,1,0,0,...,0,0,0,0,0,0,0,1.667707,49,4


## 2.2 Coefficients, Fitted Values, and Residuals

## Step 3 – Fitted Values, Residuals, and Goodness of Fit

**Fitted values:** $\hat{y}_i = \hat{\beta}_0 + \hat{\beta}_1 x_i$

**OLS residuals:** $\hat{u}_i = y_i - \hat{y}_i$

**Decomposition of total variation:**
$$SST = SSE + SSR$$
$$\underbrace{\sum_i(y_i-\bar{y})^2}_{SST} = \underbrace{\sum_i(\hat{y}_i-\bar{y})^2}_{SSE} + \underbrace{\sum_i\hat{u}_i^2}_{SSR}$$

**R-squared** (fraction of variation explained by the model):
$$R^2 = \frac{SSE}{SST} = 1 - \frac{SSR}{SST} \in [0,1]$$

In [3]:
print(f"Intercept (β₀): {model.params['Intercept']:.4f}")
print(f"Slope (β₁):     {model.params['educ']:.4f}")
print(f"\nFirst 5 fitted values:\n{model.fittedvalues.head()}")
print(f"\nFirst 5 residuals:\n{model.resid.head()}")
print(f"\nSum of residuals ≈ 0: {model.resid.sum():.6f}")

Intercept (β₀): -0.9049
Slope (β₁):     0.5414

First 5 fitted values:
0    5.050100
1    5.591459
2    5.050100
3    3.426022
4    5.591459
dtype: float64

First 5 residuals:
0   -1.950100
1   -2.351459
2   -2.050100
3    2.573978
4   -0.291459
dtype: float64

Sum of residuals ≈ 0: 0.000000


## 2.3 Goodness of Fit

## Step 4 – Log Transformations

Log-level model (semi-elasticity):
$$\log(y_i) = \beta_0 + \beta_1 x_i + u_i$$
Interpretation: a one-unit increase in $x$ changes $y$ by approximately $100\beta_1\%$.

Log-log model (constant elasticity):
$$\log(y_i) = \beta_0 + \beta_1 \log(x_i) + u_i$$
Interpretation: a 1% increase in $x$ changes $y$ by $\beta_1\%$.

> Use `np.log()` for natural logarithm. Note: requires $y > 0$.

In [ ]:
print(f"R-squared:  {model.rsquared:.4f}")
print(f"Adj. R²:    {model.rsquared_adj:.4f}")
print(f"SSR (resid): {model.ssr:.2f}")
print(f"SST (total): {(df['wage'] - df['wage'].mean()).pow(2).sum():.2f}")

## 2.4 Nonlinearities — Log Transformation

## Step 5 – Monte Carlo Verification of OLS Properties

Under SLR.1–SLR.5, OLS is **BLUE** (Best Linear Unbiased Estimator):

$$E(\hat{\beta}_1) = \beta_1 \quad \text{(unbiasedness)}$$

$$\text{Var}(\hat{\beta}_1) = \frac{\sigma^2}{\sum_i(x_i-\bar{x})^2} = \frac{\sigma^2}{n\cdot s_x^2} \quad \text{(minimum variance)}$$

We verify this by simulation: repeat the DGP many times and check that the average $\hat{\beta}_1$ converges to the true $\beta_1$.

In [ ]:
df['log_wage'] = np.log(df['wage'])
model_log = smf.ols('log_wage ~ educ', data=df).fit()
print(model_log.summary())
print(f"\nInterpretation: 1 extra year of education → "
      f"{model_log.params['educ']*100:.2f}% higher wage")

## OLS Regression Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Level
axes[0].scatter(df['educ'], df['wage'], alpha=0.3, color='steelblue')
axes[0].plot(df['educ'], model.fittedvalues, 'r-', lw=2)
axes[0].set_xlabel('Education'); axes[0].set_ylabel('Wage')
axes[0].set_title('Wage ~ Education (Level)')

# Log
axes[1].scatter(df['educ'], df['log_wage'], alpha=0.3, color='darkorange')
axes[1].plot(df['educ'], model_log.fittedvalues, 'r-', lw=2)
axes[1].set_xlabel('Education'); axes[1].set_ylabel('Log(Wage)')
axes[1].set_title('Log(Wage) ~ Education')

plt.tight_layout()
plt.savefig(f"{output_dir}/fig1_simple_ols.png", dpi=150)
plt.savefig(os.path.join(output_dir, 'fig_Chapter_02_Simple_Regression_01.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2.7 Monte Carlo Simulation of OLS

In [ ]:
np.random.seed(123)
n, n_sims = 100, 2000
true_b0, true_b1, sigma = 1.0, 0.5, 2.0

betas = []
for _ in range(n_sims):
    x = np.random.uniform(0, 10, n)
    u = np.random.normal(0, sigma, n)
    y = true_b0 + true_b1 * x + u
    temp_df = pd.DataFrame({'y': y, 'x': x})
    m = smf.ols('y ~ x', data=temp_df).fit()
    betas.append(m.params['x'])

plt.figure(figsize=(8, 4))
plt.hist(betas, bins=50, edgecolor='black', color='steelblue', density=True)
plt.axvline(true_b1, color='red', lw=2, label=f'True β₁={true_b1}')
plt.axvline(np.mean(betas), color='green', lw=2, linestyle='--',
            label=f'Mean β̂₁={np.mean(betas):.3f}')
plt.legend(); plt.title('Monte Carlo: Sampling Distribution of β̂₁')
plt.savefig(f"{output_dir}/fig2_monte_carlo_ols.png", dpi=150)
plt.savefig(os.path.join(output_dir, 'fig_Chapter_02_Simple_Regression_02.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Bias: {np.mean(betas) - true_b1:.4f}")
print(f"Std Dev of β̂₁: {np.std(betas):.4f}")